In [ ]:
from THBSplines.src.hierarchical_space import HierarchicalSpace
from THBSplines.src.cartesian_mesh import CartesianMesh
import numpy as np
import scipy.sparse as sp
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

import cffi
import numba
import numba.core.typing.cffi_utils as cffi_support
from dolfinx.jit import ffcx_jit
from dolfinx import default_real_type, default_scalar_type, geometry
rtype = default_real_type
dtype = default_scalar_type
import ufl
from ffcx.codegeneration.utils import empty_void_pointer
from ffcx.codegeneration.utils import numba_ufcx_kernel_signature as ufcx_signature

import numpy.typing as npt

In [ ]:
def refine(knots: npt.ArrayLike, p: int, n_times: int=1)->npt.NDArray:
    """Given `knots`, returns its dyadic refinement with multiplicity `p+1`
    at the extremities."""
    knots: npt.NDArray[np.float_] = np.asarray(knots)
    mult_left: int = np.searchsorted(knots, knots[0], side='right')
    mult_right: int = len(knots) - np.searchsorted(knots, knots[-1], side='left')
    pad_left: int = max(0, p + 1 - mult_left)
    pad_right: int = max(0, p + 1 - mult_right)
    if pad_left > 0 or pad_right > 0:
        knots = np.concatenate((
            np.full(pad_left, knots[0], dtype=knots.dtype),
            knots,
            np.full(pad_right, knots[-1], dtype=knots.dtype)
        ))
    if n_times == 0:
        return knots
    
    # Find indices where the knot value changes
    jump_idx: npt.NDArray[np.int_] = np.where(knots[1:] > knots[:-1])[0]
    left_vals = knots[jump_idx]
    right_vals =knots[jump_idx + 1]
    num_new_points = (1<<n_times)-1
    fractions = np.linspace(0.,1.,num_new_points+2)[1:-1]
    new_points = left_vals[:, None] + (right_vals - left_vals)[:, None] * fractions[None, :]
    new_points = new_points.ravel()
    insert_positions = np.repeat(jump_idx + 1, num_new_points)
    
    return np.insert(knots, insert_positions, new_points)
        

In [ ]:
p0 = 2
knots1 = np.array([-1., 0., 1.], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=1)
log_initial_mesh_size = np.log2(np.max(np.diff(knots1)))
knots2 = knots1# np.array([0,0, 0., 1, 1., 1], dtype=np.float64)
err_cells = {}
hs = HierarchicalSpace(knots=[knots1, knots2], degrees=[p0])

In [ ]:
for level, cells in err_cells.items():
    hs.refine_in_rectangles(cells, level, refine_neighbours=True)
    # hs.refine_in_rectangle(np.array([[-1, 0], [-1, 0]]), 0)
    # hs.refine_in_rectangle(np.array([[-0.5, .5], [-0.5, 0.5]]), 0)
    # hs.refine_in_rectangle(np.array([[-0.,1], [-0, 1]]), 0)

    # hs.refine_in_rectangle(np.array([[-1, -0.5], [-1, 0.5]]), 0)
    # hs.refine_in_rectangle(np.array([[-1, 0.5], [-1, -0.5]]), 0)
    # hs.refine_in_rectangle(np.array([[-0.5, 0.], [-1, 1.]]), 0)
    # hs.refine_in_rectangle(np.array([[0.5, 1.], [-0.5, 0.]]), 0)
    # hs.refine_in_rectangle(np.array([[-0.5,0.25], [-0.5, 0.25]]), 1)
    # hs.refine_in_rectangle(np.array([[-0.25,0.5], [-0.25, 0.5]]), 1)

    # hs.refine_in_rectangle(np.array([[-0.75, 0.], [-0.75, 0.]]), 1)
    # hs.refine_in_rectangle(np.array([[-0.75, 0.25], [-0.5, 0.]]), 1)
    # hs.refine_in_rectangle(np.array([[-0.25, .5], [-0.25, 0.5]]), 1)
    # hs.refine_in_rectangle(np.array([[0., .75], [-0., 0.75]]), 1)
    # hs.refine_in_rectangle(np.array([[-0.5, -0.125], [-0.5, -0.125]]), 2)
    # hs.refine_in_rectangle(np.array([[-0.25, .125], [-0.25, 0.125]]), 2)
    # hs.refine_in_rectangle(np.array([[-0.125, .25], [-0.125, 0.25]]), 2)
    # hs.refine_in_rectangle(np.array([[0.125, .5], [0.125, 0.5]]), 2)


hs.hmesh.plot_cells()

In [ ]:
total_active_cells = sum(len(hs.hmesh.aelem_level[l]) for l in range(hs.nlevels))

all_cells = np.empty((4*total_active_cells, 2), dtype=np.float64) # will have coarser cells on top and finer on bottom
thb_operators: dict[tuple[int, int], sp.csr_array] = {}
N_max = 0 # maximum amount of dofs in a cell
current_idx = 0
for l in range(hs.nlevels):
    active_cells_l = hs.hmesh.aelem_level[l]
    if len(active_cells_l)==0:
        continue
   
    thb_operators_list = hs.local_multi_level_extraction_operator2(active_cells_l, l, l)
    thb_operators.update({(l, cell): op for cell, op in zip(active_cells_l, thb_operators_list)})
    
    if thb_operators_list:
        level_max = max(op.shape[0] for op in thb_operators_list)
        N_max = max(N_max, level_max)

    mesh = CartesianMesh(hs.hmesh.one_d_indices[l], len(hs.hmesh.one_d_indices[l]))
    my_cells_l = mesh.cells[active_cells_l]

    n_cells = len(my_cells_l)
    x_coords = my_cells_l[:, 0, :]
    y_coords = my_cells_l[:, 1, :] 

    start, end = current_idx, current_idx+(4*n_cells)

    view = all_cells[start:end]
    #points = np.zeros((4 * len(my_cells_l), 2), dtype=np.float64)
    view[::4] = np.column_stack((x_coords[:, 0], y_coords[:, 0]))  # Bottom-left
    view[1::4] = np.column_stack((x_coords[:, 1], y_coords[:, 0]))  # Bottom-right
    view[2::4] = np.column_stack((x_coords[:, 0], y_coords[:, 1]))  # Top-left
    view[3::4] = np.column_stack((x_coords[:, 1], y_coords[:, 1]))  # Top-right

    current_idx=end
pass
del mesh # free this big object

In [ ]:
# err = all_cells[coordinates[162]]
# print(err)
# err_cell = np.zeros((2,2))
# err_cell[0, :] = err[:2, 0]
# err_cell[1, :] = err[1:3, 1]
# print(err_cell)

In [ ]:
all_cells = np.array(all_cells).reshape(-1, 2)

coordinates = np.arange(len(all_cells), dtype=np.int32).reshape(-1, 4)
coordinate_element = basix.ufl.element("Q", "quadrilateral", 1, shape=(2,))
disconnected_mesh = dolfinx.mesh.create_mesh(MPI.COMM_WORLD, cells=coordinates, e=coordinate_element, x=all_cells)

In [ ]:
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

plotter = pyvista.Plotter()
plotter.add_mesh(grid.shrink(.9), show_edges=False, color="lightblue")
plotter.view_xy()
plotter.show(jupyter_backend="static")
print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")

u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
f = dolfinx.fem.Function(V)
sigma = 0.4
# Ricker wavelet
#f.interpolate(lambda x: 1./(np.pi*sigma**4)*(1.-0.5*((x[0]**2+x[1]**2)/sigma**2)) * np.exp(-(x[0]**2+x[1]**2)/(2.*sigma**2)))
#f_sq_integral = 38.856187278294755 #[-3,3]^2, sigma=0.4
#f.interpolate(lambda x: np.cos(np.pi*x[0])*np.sin(np.pi*x[1]))
#f_sq_integral = 1.
f.interpolate(lambda x: 0.001*x[1]**2-x[0]*3.*x[1]+0.7 -0.67*x[0])
f_sq_integral = 6.5604
#f.interpolate(lambda x: 2./(3.*np.exp(np.sqrt((10.*x[0]-3.)**2 + (10*x[1]-3)**2))) + 
#               2./(3.*np.exp(np.sqrt((10.*x[0]+3.)**2 + (10*x[1]+3)**2))) + 
#               2./(3.*np.exp(np.sqrt((10.*x[0])**2 + (10*x[1])**2))))
#f_sq_integral = 0.024260459017934315
a0 = ufl.inner(u, v) * ufl.dx
f0 = ufl.inner(f, v)*ufl.dx

msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcxf0, _, _ = ffcx_jit(msh.comm, f0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernelf0 = getattr(ufcxf0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ffi = cffi.FFI()

In [ ]:
dofmap, dummy_dof_index = hs.build_global_dof_map()
dummy_dof_index += 1 
# does not exist, and we will map unwanted contributions to this function

def build_dofmap(hs, funcs=None, my_dict={}, level=0):
    if funcs is None:
        funcs = np.arange(hs.level_spaces[0].nfuncs_total, dtype=np.int32)
    #print(f"funcs = {funcs}")
    for func in funcs:
        # print(f"func = {func}")
        if (level, func) not in my_dict and func in hs.truly_active[level]:
            my_dict[level, func] = len(my_dict)
        pass
        if level<hs.nlevels-1:
            children = hs.level_spaces[level].get_children_functions(func)[0]
            my_dict = build_dofmap(hs, children, my_dict, level+1)
    return my_dict
    
#dofmap = build_dofmap(hs)
#dummy_dof_index = len(dofmap)

In [ ]:
num_control_points_with_dummy = dummy_dof_index+1

num_cells = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_global

# which BSpline functions are active on each cell, padded with the dummy index.
padded_cells_to_dofs = np.full((num_cells, N_max), dummy_dof_index, dtype=np.int32)

# Find the maximum local index for each level to size arrays
max_local_indices = [0] * hs.nlevels
for l, local_idx in dofmap.keys():
    if local_idx > max_local_indices[l]:
        max_local_indices[l] = local_idx
    pass
pass

# convert the dictionary into a list of numpy arrays for faster lookup
dofmap_arrays = [np.full(size + 1, dummy_dof_index, dtype=np.int32) for size in max_local_indices]
for (l, local_idx), global_idx in dofmap.items():
    dofmap_arrays[l][local_idx] = global_idx


cell_index = 0
for l in range(hs.nlevels):
    active_cells_l = hs.hmesh.aelem_level[l]
    for cell in active_cells_l:
        
        # Get the local functions on the cell
        active_funcs_dict: dict[int, npt.NDArray[np.int_]] = hs.get_all_active_functions_on_cell(l, cell)
        
            
        #  Vectorized conversion from local to global indices
        mapped_arrays = [
            dofmap_arrays[ll][local_funcs] for ll, local_funcs in active_funcs_dict.items() if len(local_funcs) > 0
        ]
        if l<0:
            print(f"level={l}, cell={cell}")
            print(f"active funcs = \n {active_funcs_dict}")
            print(f"mapped arrays = \n {mapped_arrays} \n\n")
        
        #
        if mapped_arrays: # Check if there are actually active functions
            global_dofs = np.concatenate(mapped_arrays)
            n_dofs = len(global_dofs)
            padded_cells_to_dofs[cell_index, :n_dofs] = global_dofs
            
        cell_index += 1


In [ ]:
new_pctd = padded_cells_to_dofs
for i in range(len(new_pctd)):
    row = new_pctd[i, :]
    no_dummy = row[row<dummy_dof_index]
    my_median = int(np.median(no_dummy))
    new_pctd[i, :][new_pctd[i, :]==dummy_dof_index] = my_median
# print(padded_cells_to_dofs)

In [ ]:
#new_pctd=np.arange(new_pctd.size, dtype=np.int32).reshape(num_cells, -1)
#padded_cells_to_dofs = new_pctd

In [ ]:
M = hs._bezier_to_legendre(degree = p0)
S_indices = np.arange(p0+1, dtype=np.float64)
# scaling for unnormalised Legendre basis polynomials
S_inv = (1./np.sqrt(2.*S_indices+1.))*np.identity(p0+1, dtype=np.float64) # Scaling factor, since fenicsx uses orthonormal legendre polynomials
T = np.asfortranarray(np.kron(M, M).T @ np.kron(S_inv, S_inv), dtype=dtype)
local_dofs_size = T.shape[1]

operator_shape = (N_max, local_dofs_size)
# Create a custom space that holds the content of each matrix for the relevant cell.
# degree 0 because the value is constant over each cell
C_space = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0, operator_shape))
C_func = dolfinx.fem.Function(C_space, dtype=dtype)

num_cells_local = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_local
indices = np.arange(num_cells_local, dtype=np.int32)
# To make sure that each matrix is assigned to the correct cell
midpoints: npt.NDArray[np.float_] = dolfinx.mesh.compute_midpoints(disconnected_mesh, disconnected_mesh.topology.dim, indices)


In [ ]:
c_values = C_func.x.array.reshape((-1, N_max, T.shape[1]))
for local_idx, midpoint in enumerate(midpoints):
    #print(f"midpoint = {midpoint}")
    level, idx = hs.hmesh.find_active_cell(midpoint[:hs.dim])
    #print(f"midpoint = {midpoint}, level={level}, idx={idx}")
    mat: sp.csr_array = thb_operators[level, idx] @ hs.level_spaces[level].get_bezier_operator(idx)
    # print(mat.shape)
    real_k, n_cols = mat.shape

    if real_k<N_max:
        padding_size = N_max - real_k
        # add rows of zeros below the result
        # new_indptr: npt.NDArray[np.int_] = np.concatenate([
        #     mat.indptr, 
        #     np.full(padding_size, mat.indptr[-1], dtype=mat.indptr.dtype)
        # ])
        # mat_padded = sp.csr_array(
        #     (mat.data, mat.indices, new_indptr), 
        #     shape=(N_max, n_cols)
        # )
        # Ci = mat_padded
        mat_padded = np.vstack((mat, np.zeros((padding_size, mat.shape[1])) ))
        Ci = mat_padded
    else:
        Ci = mat
    
    # is a view of C_func.x.array, therefore we modify the content of C_func.x.array
    # No new array is created, the matrix->cell mapping is done here.
    c_values[local_idx, :, :] = Ci@T
C_func.x.scatter_forward()

In [ ]:
#hs.local_multi_level_extraction_operator(8, 1, 1)

In [ ]:
num_control_points = np.max(new_pctd)+1
my_index_map = dolfinx.common.IndexMap(comm=disconnected_mesh.comm, 
                                       local_size=num_control_points)

dummy_element = basix.ufl.element(
    family="DG", 
    cell="quadrilateral", 
    degree=0, 
    shape=(N_max,)
)
dummy_space = dolfinx.fem.functionspace(mesh=disconnected_mesh, 
                                        element=dummy_element)
element_layout = dummy_space.dofmap.dof_layout
cpp_element = dummy_space.element._cpp_object

dof_indices = padded_cells_to_dofs.ravel().astype(np.int32)
offsets = (np.arange(num_cells + 1, dtype=np.int32) * N_max).astype(np.int32)

adj = dolfinx.cpp.graph.AdjacencyList_int32(data=dof_indices, 
                                            offsets=offsets)

dofmap = dolfinx.cpp.fem.DofMap(
    element_dof_layout=element_layout, 
    index_map=my_index_map,  
    index_map_bs=1, 
    dofmap=adj, 
    bs=1
)

V_spline_cpp = dolfinx.cpp.fem.FunctionSpace_float64(
    mesh=disconnected_mesh._cpp_object, 
    element=cpp_element, 
    dofmap=dofmap
)

V_spline = dolfinx.fem.FunctionSpace(
    mesh=disconnected_mesh, 
    element=dummy_element, 
    cppV=V_spline_cpp
)

In [ ]:
PADDED_DOFS = N_max
LOCAL_DOFS = local_dofs_size

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)  # type: ignore
def tabulate_A(A_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):

    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # returns a view over the original array A_
    # This has to be larger since we are working with padded arrays. Irrelevant dofs are mapped to a dummy location
    A = numba.carray(A_, (PADDED_DOFS, PADDED_DOFS), dtype=dtype)

    # Get the operator (TRUNC @ C_{B->BS} @ (C_{L->B}.T) @ S^{-1}) for this cell
    # TRUNC has shape (PADDED_DOFS, LOCAL_DOFS) and all other matrices have shape (LOCAL_DOFS, LOCAL_DOFS)
    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)
    
    # Tabulate all sub blocks locally
    # This matrix is formed via the Legendre elements on a quadrilateral of degree p0,
    # therefore this has the shape (LOCAL_DOFS, LOCAL_DOFS)
    A0 = np.zeros((LOCAL_DOFS, LOCAL_DOFS), dtype=dtype)
    kernela0(
        ffi.from_buffer(A0),
        w_,# weights. Ignored for the L2 approximation because 
        # a0 does not depend on f and no value is looked up at the quadrature points
        # This is not very robust, one must be careful when calling this function
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )
    
    A[:, :] = G@A0@(G.T) #np.append(np.append(A0, np.zeros((PADDED_DOFS-LOCAL_DOFS, LOCAL_DOFS)), axis=0), 
                       # np.zeros((PADDED_DOFS, PADDED_DOFS-LOCAL_DOFS)), axis=1)
 # 

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)
def tabulate_b(b_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):
    
    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # evaluation of f using padded THB-Splines
    b = numba.carray(b_, (PADDED_DOFS,), dtype=dtype)

    total_coeffs = LOCAL_DOFS*(PADDED_DOFS+1)
    # Evaluation of f at relevant points + values of change of basis matrix G
    w_flat = numba.carray(w_, (total_coeffs), dtype=dtype)

    # The following line breaks in case we do not pass [f._cpp_object, C_func._cpp_object] in this
    # exact order or if f does not have the correct amount of dofs.
    G = w_flat[LOCAL_DOFS:].reshape((PADDED_DOFS, LOCAL_DOFS))

    b0 = np.zeros((LOCAL_DOFS,), dtype=dtype)
    kernelf0(ffi.from_buffer(b0),
        w_, # This line only works because we specified that b0 is of size LOCAL_DOFS,
        # which happens to also be the size of f._cpp_object
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )
    # G@b0
    b[:] = G@b0# np.append(b0, np.zeros(PADDED_DOFS-LOCAL_DOFS))# 

In [ ]:
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object
                    ], # weights w_, holds C@T
              constants=[],
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_b.address, cells, np.array([0,1], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of tabulate_b
        coefficients=[f._cpp_object, C_func._cpp_object], # holds the evaluations of f at the correct points, as well as C@T
        constants=[], need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
from dolfinx.fem.petsc import assemble_matrix, assemble_vector
from petsc4py import PETSc
# a_form = dolfinx.fem.form(a0)
A_cond = assemble_matrix(a_cond, bcs=[])
A_cond.assemble()
# A_mat = A_cond
# A_mat.setValue(dummy_dof_index, dummy_dof_index, 1., addv=PETSc.InsertMode.INSERT_VALUES)
# A_mat.assemble()
# A_mat.assemblyBegin()
# A_mat.assemblyEnd()

L_cond = assemble_vector(l_cond)
# L_cond[dummy_dof_index]=0.
# L_cond.assemblyBegin()
# L_cond.assemblyEnd()
L_cond.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)


In [ ]:
# Extract the CSR (Compressed Sparse Row) arrays from PETSc
indptr, indices, data = A_cond.getValuesCSR()

# Get the global size of the matrix
shape = A_cond.getSize()

# Create a SciPy CSR matrix
A_scipy = sp.csr_array((data, indices, indptr), shape=shape)

print(f"Matrix shape: {A_scipy.shape}")
print(f"Number of non-zeros: {A_scipy.nnz}")

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))
# plt.spy plots the non-zero entries of a matrix
plt.spy(A_scipy, markersize=2, color='lightseagreen')
plt.title("Sparsity Pattern of THB-Spline mass Matrix")
plt.show()

In [ ]:
# ew1, ev = sp.linalg.eigsh(A_scipy, which='LM')
# ew2, ev = sp.linalg.eigsh(A_scipy, sigma=1e-8)
# print(abs(ew1).max()/abs(ew2.min()))

In [ ]:
ksp = PETSc.KSP().create(A_cond.comm)
ksp.setOperators(A_cond)
#ksp.setType(PETSc.KSP.Type.CG)
#ksp.getPC().setType(PETSc.PC.Type.JACOBI)
ksp.setType(PETSc.KSP.Type.PREONLY)
ksp.getPC().setType(PETSc.PC.Type.LU)
ksp.getPC().setFactorSolverType("mumps")
u_sol = dolfinx.fem.Function(V_spline)
ksp.solve(L_cond, u_sol.x.petsc_vec)
u_sol.x.scatter_forward()
x_vec=u_sol.x.array
print(f"Solve complete. Reason: {ksp.getConvergedReason()}, Iterations: {ksp.getIterationNumber()}")

In [ ]:
#f_sq_form = dolfinx.fem.form(ufl.inner(f, f) * ufl.dx)
#f_sq_integral = dolfinx.fem.assemble_scalar(f_sq_form)
#f_sq_integral = disconnected_mesh.comm.allreduce(f_sq_integral, op=MPI.SUM)
x_dot_L = L_cond.dot(u_sol.x.petsc_vec)
l2_error_sq = f_sq_integral - x_dot_L
l2_error = np.sqrt(abs(l2_error_sq))

print(f"Squared integral of f: {f_sq_integral:.4f}")
print(f"x^T L: {x_dot_L:.8f}")
print(f"L2 Error: {l2_error:.4e}")
print(f"relative error = {l2_error/f_sq_integral:.2e}")

In [ ]:
u_dg = dolfinx.fem.Function(V)

# Map the global B-spline coefficients back to local Legendre coefficients
for local_idx in range(num_cells_local):
    # Get global B-spline dof indices for this cell
    spline_dofs = padded_cells_to_dofs[local_idx]
    
    # Extract the B-spline coefficients for this cell
    u_spline_local = x_vec[spline_dofs]
    
    # Get the local transformation matrix G for this cell
    G = c_values[local_idx, :, :]
    
    # Transform B-spline to DG: mathematically, the kernel does A = G @ A0 @ G.T
    # This implies the coefficient mapping is u_dg = G.T @ u_spline
    u_dg_local = G.T @ u_spline_local
    
    # Assign to the standard DG function
    dg_dofs = V.dofmap.cell_dofs(local_idx)
    u_dg.x.array[dg_dofs] = u_dg_local

u_dg.x.scatter_forward()

# Compute exact L2 error using FEniCSx standard UFL
error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * ufl.dx)
error_sq = dolfinx.fem.assemble_scalar(error_form)
exact_l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_sq, op=MPI.SUM))

print(f"Exact L2 Error (via DG projection): {exact_l2_error:.2e}")
print(f"Relative error = {exact_l2_error/f_sq_integral:.2e}")

In [ ]:
# Create a DG0 space (one value per cell)
V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v = ufl.TestFunction(V_error)

# Define the local L2 error form: integral of (f - u_dg)^2 per cell
# Note: We multiply by the test function 'v' to pick out each cell's contribution
local_error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * v * ufl.dx)

# Assemble the vector (this gives us the squared error per cell)
local_error_vector = dolfinx.fem.assemble_vector(local_error_form)
local_error_vector.scatter_forward()

# Take the square root to get the L2 norm per cell
# (Using .array for NumPy manipulation)
cell_errors = np.sqrt(local_error_vector.array)

# Calculate the threshold for the top 25%
proportion = 0.25
num_cells = len(cell_errors)
top_percent_count = int(max(1, num_cells *proportion))

# Get the indices of the cells sorted by error (ascending)
sorted_indices = np.argsort(cell_errors)

# Extract the indices of the top 25%
top_error_indices = sorted_indices[-top_percent_count:]

print(f"Top {proportion*100}% error threshold: {cell_errors[sorted_indices[-top_percent_count]]:.4e}")
print(f"Indices of top {proportion*100}% error cells: {top_error_indices[:10]}")

In [ ]:
my_dict = {}

In [ ]:
err_cells = {}
for i in top_error_indices:
    err = all_cells[coordinates[i]]
    err_cell = np.zeros((2,2))
    err_cell[0, :] = err[:2, 0]
    err_cell[1, :] = err[1:3, 1]
    level = int(-np.log2(err_cell[0,1]-err_cell[0,0])+log_initial_mesh_size)
    #err_cells.append(tuple((err_cell, int(level))))
    if level not in err_cells:
        err_cells[level] = [err_cell]
    else:
        err_cells[level].append(err_cell)
    

In [ ]:
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# grid.point_data["v"] = f.x.array.real # Use interpolated function for demo
# grid.point_data["u"] = u_dg.x.array.real

# plotter = pyvista.Plotter()
# plotter.add_mesh(grid.shrink(0.9), scalars="u", show_edges=True)
# plotter.view_xy()
# plotter.show(jupyter_backend="static")

In [ ]:
ksp.destroy()
A_cond.destroy()
L_cond.destroy()